In [10]:
import sqlite3
import sys

In [11]:
DB_PATH = "trading.db"

In [12]:
def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

### Print all rows as dictionaries

In [42]:
import json

def analyze_trades():
    conn = get_conn()
    trades = conn.execute("SELECT * FROM trade_logs").fetchall()
    conn.close()

    return json.dumps(
        [dict(row) for row in trades],
        indent=4
    )

print(analyze_trades())




[
    {
        "id": 4702,
        "user_id": 2,
        "symbol": "XAUUSD",
        "action": "BUY",
        "lot_size": 0.2,
        "entry_price": 4504.66,
        "stop_loss": 0.0,
        "take_profit": 0.0,
        "ticket": 240963344,
        "status": "CLOSED",
        "profit": 18.0,
        "h1_bias": "N/A",
        "ai_decision": "MT5",
        "m5_zone": "N/A",
        "opened_at": "2026-05-27T05:13:53+00:00",
        "closed_at": "2026-05-27T05:33:59+00:00"
    },
    {
        "id": 4703,
        "user_id": 2,
        "symbol": "XAUUSD",
        "action": "BUY",
        "lot_size": 0.2,
        "entry_price": 4501.88,
        "stop_loss": 0.0,
        "take_profit": 0.0,
        "ticket": 240967101,
        "status": "CLOSED",
        "profit": 21.4,
        "h1_bias": "N/A",
        "ai_decision": "MT5",
        "m5_zone": "N/A",
        "opened_at": "2026-05-27T05:40:54+00:00",
        "closed_at": "2026-05-27T05:50:48+00:00"
    },
    {
        "id": 4704,
        "u

### Convert to a Pandas DataFrame

In [40]:
import pandas as pd

def analyze_trades():
    conn = get_conn()

    df = pd.read_sql_query(
        "SELECT * FROM trade_logs",
        conn
    )

    conn.close()
    return df

df = analyze_trades()

trade = df.iloc[0]

for key, value in trade.items():
    print(f"{key:15}: {value}")

id             : 4702
user_id        : 2
symbol         : XAUUSD
action         : BUY
lot_size       : 0.2
entry_price    : 4504.66
stop_loss      : 0.0
take_profit    : 0.0
ticket         : 240963344
status         : CLOSED
profit         : 18.0
h1_bias        : N/A
ai_decision    : MT5
m5_zone        : N/A
opened_at      : 2026-05-27T05:13:53+00:00
closed_at      : 2026-05-27T05:33:59+00:00


In [50]:
conn = get_conn()

columns = conn.execute(
    "PRAGMA table_info(trade_logs)"
).fetchall()

print("\nTRADE_LOGS TABLE SCHEMA")
print("=" * 80)

for col in columns:
    c = dict(col)

    print(
        f"Column: {c['name']:<20} "
        f"Type: {c['type']:<10} "
        f"Primary Key: {bool(c['pk'])}"
    )

conn.close()


TRADE_LOGS TABLE SCHEMA
Column: id                   Type: INTEGER    Primary Key: True
Column: user_id              Type: INTEGER    Primary Key: False
Column: symbol               Type: TEXT       Primary Key: False
Column: action               Type: TEXT       Primary Key: False
Column: lot_size             Type: REAL       Primary Key: False
Column: entry_price          Type: REAL       Primary Key: False
Column: stop_loss            Type: REAL       Primary Key: False
Column: take_profit          Type: REAL       Primary Key: False
Column: ticket               Type: INTEGER    Primary Key: False
Column: status               Type: TEXT       Primary Key: False
Column: profit               Type: REAL       Primary Key: False
Column: h1_bias              Type: TEXT       Primary Key: False
Column: ai_decision          Type: TEXT       Primary Key: False
Column: m5_zone              Type: TEXT       Primary Key: False
Column: opened_at            Type: TIMESTAMP  Primary Key: False
C

In [23]:
def status():
    """Show row counts for all tables."""
    conn = get_conn()
    tables = ["users", "mt5_credentials", "trade_logs", "ai_decision_logs",
              "error_logs", "performance_metrics", "user_settings"]
    print("\n  TABLE                    ROWS")
    print("  " + "-" * 35)
    for t in tables:
        try:
            count = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
            print(f"  {t:<25} {count}")
        except Exception:
            print(f"  {t:<25} (not found)")
    conn.close()

status()


  TABLE                    ROWS
  -----------------------------------
  users                     2
  mt5_credentials           1
  trade_logs                49
  ai_decision_logs          0
  error_logs                0
  performance_metrics       0
  user_settings             1


In [24]:
def clear_decisions():
    """Clear ai_decision_logs table."""
    conn = get_conn()
    cur = conn.execute("DELETE FROM ai_decision_logs")
    conn.commit()
    print(f"Deleted {cur.rowcount} rows from ai_decision_logs")
    conn.close()

clear_decisions()


Deleted 0 rows from ai_decision_logs


In [25]:
def clear_trades():
    """Clear trade_logs table."""
    conn = get_conn()
    cur = conn.execute("DELETE FROM trade_logs")
    conn.commit()
    print(f"Deleted {cur.rowcount} rows from trade_logs")
    conn.close()


clear_trades()

Deleted 49 rows from trade_logs


In [26]:
def clear_errors():
    """Clear error_logs table."""
    conn = get_conn()
    cur = conn.execute("DELETE FROM error_logs")
    conn.commit()
    print(f"Deleted {cur.rowcount} rows from error_logs")
    conn.close()

clear_errors()

Deleted 0 rows from error_logs


In [27]:
def clear_all():
    """Clear all log tables (keeps users and credentials)."""
    conn = get_conn()
    for table in ["ai_decision_logs", "trade_logs", "error_logs"]:
        cur = conn.execute(f"DELETE FROM {table}")
        print(f"  Deleted {cur.rowcount} rows from {table}")
    conn.execute("UPDATE performance_metrics SET total_trades=0, winning_trades=0, "
                 "losing_trades=0, total_profit=0, max_drawdown=0, win_rate=0")
    conn.commit()
    print("  Reset performance_metrics")
    conn.close()

clear_all()

  Deleted 0 rows from ai_decision_logs
  Deleted 0 rows from trade_logs
  Deleted 0 rows from error_logs
  Reset performance_metrics


In [28]:
def vacuum():
    """Reclaim disk space after deletions."""
    conn = get_conn()
    conn.execute("VACUUM")
    conn.close()
    print("Database vacuumed (disk space reclaimed)")

vacuum()

Database vacuumed (disk space reclaimed)
